## Setup
Installing `pymupdf` for extracting text directly from the budget PDF.

In [ ]:
!pip install pymupdf -q

## Load source documents
Uploading the Chipata City Council 2024 Output-Based Annual Budget and the
2025 Bi-annual Budget Performance report — the two source PDFs for the
budget allocations table.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Confirm uploaded filenames
Checking the exact filenames Colab saved, since upload can alter naming
(e.g. appended characters) — needed to reference the files correctly in code.

In [ ]:
import os
print(os.listdir())

## Standardise filename
Renaming the budget PDF to a simpler, consistent name for use throughout
the notebook, since the uploaded filename may include extra characters.

In [ ]:
files_here = [f for f in os.listdir() if 'Chipata' in f and 'budget' in f]
if files_here:
    os.rename(files_here[0], "budget_2024.pdf")
print(os.listdir())

## Inspect document structure
Opening the budget PDF and extracting text from page 1 to confirm the file
has a native text layer (not scanned) and to understand its structure
before writing extraction logic.

In [ ]:
import pymupdf

doc = pymupdf.open("budget_2024.pdf")
print(f"Total pages: {len(doc)}")
print("--- PAGE 1 TEXT ---")
print(doc[0].get_text())

## Locate the budget tables
Scanning all pages to identify where the actual financial data sits.
Pages 1–2 contain mandate/strategy narrative; pages 3–4 cover revenue
(handled separately in the revenue_streams table); page 8 contains the
master Budget Allocation by Programme and Sub-Programme table (Table 3);
pages 11 onward break this down per individual programme (Constituency
Development, Local Governance, IDP, etc.) with Approved figures for
2022, 2023, and 2024.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200])
    print()

## Extract full text from key tables
Pulling the complete text of page 8 (master Budget Allocation by Programme
and Sub-Programme summary) and page 11 (first individual programme
breakdown — Constituency Development) to examine how figures, years, and
programme names are laid out before writing the parser.

In [ ]:
print("=== PAGE 8 FULL TEXT ===")
print(doc[7].get_text())  # index 7 = page 8

print("\n\n=== PAGE 11 FULL TEXT ===")
print(doc[10].get_text())  # index 10 = page 11

## Parse Table 3 — Budget Allocation by Programme and Sub-Programme (page 8)
The PDF's table structure extracts as flat text with columns out of visual order
(a known pymupdf limitation on multi-row-header tables). Each programme/sub-programme
is a fixed 7-line block: name, code, then 5 numeric values. In this document, only
one value per row is a real figure — the rest are placeholders ("(0)" or "-") since
prior-year comparative data isn't populated in this budget cycle. We extract that
single real value as the 2024 approved budget amount. Rows are flagged
`needs_review` since the 2022/2023 column mapping isn't reliably determined from
text order alone — this will need spot-checking against the source PDF.

In [ ]:
import pandas as pd

page8_text = doc[7].get_text()
lines = [l.strip() for l in page8_text.split('\n') if l.strip()]

# Data starts right after the header line "Estimate"
start_idx = lines.index('Estimate') + 1
data_lines = lines[start_idx:]

rows = []
i = 0
while i <= len(data_lines) - 7:
    name = data_lines[i]
    code = data_lines[i+1]

    # sanity check: code should be a short number (1-3 digits)
    if not code.replace('.', '').isdigit() or len(code) > 3:
        i += 1
        continue

    raw_values = data_lines[i+2:i+7]
    numeric_values = []
    for v in raw_values:
        v_clean = v.replace(',', '').replace('(0)', '0').replace('-', '0').strip()
        try:
            numeric_values.append(float(v_clean))
        except ValueError:
            numeric_values.append(0.0)

    approved_2024 = max(numeric_values) if numeric_values else 0.0

    rows.append({
        'record_id': f"BUD-2024-{code}",
        'council_name': 'Chipata City Council',
        'budget_year': 2024,
        'budget_period': 'annual',
        'vote_head_code': code,
        'programme_name': name,
        'sub_programme': '',
        'budget_category': 'other',
        'approved_budget_zmw': approved_2024,
        'source_document': 'budget_2024.pdf',
        'source_page': 8,
        'extraction_method': 'native_text',
        'data_quality_flag': 'needs_review',
        'notes': 'Column order inferred from single non-zero value per row; verify against source PDF'
    })
    i += 7

df_page8 = pd.DataFrame(rows)
print(f"Extracted {len(df_page8)} rows")
df_page8

## Fix duplicate record IDs and mark programme vs sub-programme
Vote head codes reset within each parent programme (e.g. code "001" appears
under multiple different programmes), so `record_id` built from code alone
isn't unique. Rebuilding it using a running row index instead. Also flagging
whether each row is a top-level programme (single-digit code, e.g. "1", "2")
or a sub-programme (3-digit code, e.g. "779", "044") based on code length,
since the source table nests sub-programmes under their parent programme.

In [ ]:
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

df_page8['programme_level'] = df_page8['vote_head_code'].apply(
    lambda c: 'programme' if len(c.strip()) <= 2 else 'sub_programme'
)

# quick sanity check
print(df_page8[['record_id', 'vote_head_code', 'programme_name', 'programme_level', 'approved_budget_zmw']].to_string())

## Check for LGEF-specific content
The assignment brief explicitly requires "LGEF utilisation" data. Searching
the full document for any mentions of LGEF (Local Government Equalisation
Fund) to confirm whether it appears as a distinct budget line or funding
source that needs separate tagging.

In [ ]:
for i in range(len(doc)):
    text = doc[i].get_text()
    if 'LGEF' in text.upper() or 'EQUALISATION' in text.upper():
        print(f"Page {i+1} mentions LGEF:")
        idx = text.upper().find('LGEF') if 'LGEF' in text.upper() else text.upper().find('EQUALISATION')
        print(text[max(0,idx-100):idx+300])
        print("---")

## LGEF finding
Page 4 shows LGEF as a revenue line item (K15.2M approved for 2024) under
"Grants" — this is the council's LGEF *income*, which belongs in the revenue
streams table, not budget allocations.
Page 8's programme-level budget table doesn't break down funding source per
programme, so a true "LGEF utilisation by programme" breakdown isn't
extractable from this document. Flagging this gap rather than fabricating
a mapping — noting it in the Data Description Paper's limitations section.

## Load and inspect bi-annual performance report
Opening the second source document to check its structure before extracting
actual expenditure figures, which will let us calculate budget variance
per programme.

In [ ]:
doc2 = pymupdf.open("2025-Bi-annual-Budget-Perfomance.pdf")
print(f"Total pages: {len(doc2)}")
for i in range(min(5, len(doc2))):
    print(f"=== Page {i+1} ===")
    print(doc2[i].get_text()[:400])
    print()

## Confirm scan status — OCR required
Native text extraction on the bi-annual performance PDF returns garbled/unreadable
characters, indicating this file is scanned (image-based), not text-native,
despite earlier assumptions. OCR is required to extract this document.

## Set up OCR pipeline for bi-annual performance report
The bi-annual budget performance PDF is scanned/image-based, so native text
extraction fails. Installing Tesseract OCR and pdf2image to convert each
page to an image and extract text via OCR instead. This will let us pull
actual expenditure figures to calculate budget variance per programme,
strengthening the "LGEF utilisation" and comprehensiveness requirements
in the assignment brief.

In [ ]:
!apt-get install -y tesseract-ocr poppler-utils -q
!pip install pytesseract pdf2image -q

## Convert scanned pages to images and run OCR
Converting each page of the bi-annual performance PDF to a high-resolution
image, then extracting text via Tesseract OCR. Starting with page 1 only
to verify OCR quality before running it across all 8 pages.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300, first_page=1, last_page=1)
ocr_text = pytesseract.image_to_string(images[0])
print(ocr_text)

## Visually inspect the scanned page
OCR output on page 1 was mostly garbled. Displaying the raw converted image
to check for rotation, skew, or resolution issues before adjusting the OCR
pipeline.

In [ ]:
images[0]

## Rotate image and re-run OCR
Page was scanned rotated (confirmed via visual inspection), which degrades
OCR accuracy significantly even on a clear, high-quality scan. Rotating
90° counter-clockwise corrects the orientation before extracting text.

In [ ]:
# Rotate 90° counter-clockwise to correct the scan orientation
rotated = images[0].rotate(-90, expand=True)
rotated

In [ ]:
ocr_text = pytesseract.image_to_string(rotated)
print(ocr_text)

## OCR successful — check document scope
Rotation fix resolved OCR quality significantly. However, this page shows
revenue performance (budget vs. actual receipts by revenue source: Local
Taxes, Fees & Charges, Grants), not programme-level expenditure. This maps
more directly to the revenue_streams table than budget_allocations.
Checking remaining pages before deciding final placement.

In [ ]:
# Process all 8 pages now that rotation angle is confirmed
all_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=300)

for i, img in enumerate(all_images):
    rotated_img = img.rotate(-90, expand=True)
    text = pytesseract.image_to_string(rotated_img)
    print(f"=== PAGE {i+1} ===")
    print(text[:400])
    print()

## Fix programme/sub-programme structure and infer budget category
Restructuring so `programme_name` always holds the parent programme (e.g.
"Constituency Development"), and `sub_programme` holds the child line item
name (e.g. "Community Projects") where applicable — matching the schema's
intended structure. Also inferring `budget_category` from programme context:
CDF-related programmes are tagged `cdf`, core administrative/support
programmes as `recurrent_expenditure`, and infrastructure/asset-heavy
programmes as `capital_expenditure` where the source text confirms this
(e.g. page 6 states Housing and Community Amenities is largely capital
assets). Rows without a clear signal are left as `other` and flagged for
manual review.

In [ ]:
# Step 1: forward-fill parent programme name into sub-programme rows
current_parent = None
fixed_programme_name = []
fixed_sub_programme = []

for _, row in df_page8.iterrows():
    if row['programme_level'] == 'programme':
        current_parent = row['programme_name']
        fixed_programme_name.append(row['programme_name'])
        fixed_sub_programme.append('')
    else:  # sub_programme
        fixed_programme_name.append(current_parent)
        fixed_sub_programme.append(row['programme_name'])

df_page8['programme_name'] = fixed_programme_name
df_page8['sub_programme'] = fixed_sub_programme

# Step 2: infer budget_category based on parent programme context
category_map = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

df_page8['budget_category'] = df_page8['programme_name'].map(category_map).fillna('other')

# quick check
print(df_page8[['programme_name', 'sub_programme', 'vote_head_code', 'budget_category', 'approved_budget_zmw']].to_string())

## Validate total against stated budget figure
Cross-checking the sum of top-level programme approved amounts against the
K150.2 million total stated in the narrative (page 5), to confirm whether
page 8's table was fully captured or continues beyond what was extracted.

In [ ]:
programme_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"Sum of top-level programmes: {programme_total:,.2f} ZMW")
print(f"Stated total in document: ~150,200,000 ZMW")
print(f"Difference: {150_200_000 - programme_total:,.2f} ZMW")

## Check for table continuation on page 9
Page 8's extracted text ends abruptly after "Primary Health Services,"
suggesting the table may continue onto page 9 with additional programmes
or sub-programmes not yet captured. Checking page 9's full text to confirm.

In [ ]:
print(doc[8].get_text())  # index 8 = page 9

## Extract remaining rows from page 9 and merge
Page 8's table continues onto page 9 after the narrative paragraph, with
3 additional programmes/sub-programmes (District Health Coordination,
Veterinary services, Transport Services) not yet captured. The page also
confirms a Head Total of 150,164,034 ZMW, matching the narrative figure
and validating completeness once these rows are added. Manually adding
these since the table fragment is small and mixed with narrative text
(same parsing logic as page 8 would misfire on this fragment).

In [ ]:
extra_rows = [
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '002', 'programme_name': 'District Health servcies',
     'sub_programme': 'District Health Coordination', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 3968017.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '14', 'programme_name': 'Veterinary services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Veterinary services',
     'sub_programme': 'Animal Health Extension Services', 'budget_category': 'other',
     'approved_budget_zmw': 189173.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '15', 'programme_name': 'Transport Services',
     'sub_programme': '', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
    {'record_id': None, 'council_name': 'Chipata City Council', 'budget_year': 2024,
     'budget_period': 'annual', 'vote_head_code': '001', 'programme_name': 'Transport Services',
     'sub_programme': 'Road Transport', 'budget_category': 'other',
     'approved_budget_zmw': 3708668.0, 'source_document': 'budget_2024.pdf', 'source_page': 9,
     'extraction_method': 'native_text', 'data_quality_flag': 'needs_review',
     'notes': 'Continuation of page 8 table, extracted from page 9'},
]

import pandas as pd
df_page8 = pd.concat([df_page8, pd.DataFrame(extra_rows)], ignore_index=True)

# rebuild record_id sequentially for all rows
df_page8['record_id'] = [f"BUD-2024-{i:03d}" for i in range(1, len(df_page8) + 1)]

# re-validate total
top_level_total = df_page8[df_page8['sub_programme'] == '']['approved_budget_zmw'].sum()
print(f"New top-level total: {top_level_total:,.2f} ZMW")
print(f"Head Total stated in document: 150,164,034.00 ZMW")
print(f"Difference: {150_164_034 - top_level_total:,.2f} ZMW")
print(f"\nTotal rows now: {len(df_page8)}")

## Improve OCR quality on the Payments page (page 2)
Re-converting page 2 at higher resolution (400 DPI) with grayscale and
contrast enhancement to improve OCR accuracy on the scanned Payments table,
since the initial pass at standard resolution produced too many blanks
and garbled values to use reliably.

In [ ]:
from PIL import ImageEnhance

# Re-convert just page 2 at higher DPI
page2_images = convert_from_path("2025-Bi-annual-Budget-Perfomance.pdf", dpi=400, first_page=2, last_page=2)
page2_rotated = page2_images[0].rotate(-90, expand=True)

# Preprocess: grayscale + boost contrast
page2_gray = page2_rotated.convert('L')
enhancer = ImageEnhance.Contrast(page2_gray)
page2_enhanced = enhancer.enhance(2.0)

page2_enhanced

## Extract text via OCR with table-optimized settings
Running Tesseract with `--psm 6` (assumes a uniform block of text, better
suited to tabular layouts than the default page-segmentation mode).

In [ ]:
ocr_text_page2 = pytesseract.image_to_string(page2_enhanced, config='--psm 6')
print(ocr_text_page2)

## Crop table region and retry OCR
Full-page OCR remains unreliable on individual payment line items due to
scan quality and dense table formatting. Cropping to isolate just the
table body (excluding headers/footers) and retrying at higher resolution
with column-oriented PSM mode, to see if this improves accuracy before
deciding whether line-item detail is extractable at all.

In [ ]:
# View full image with dimensions to identify crop coordinates
width, height = page2_enhanced.size
print(f"Image size: {width} x {height}")
page2_enhanced

## Crop to isolate table region
Cropping the image to the estimated table boundaries (excluding header
and footer/signature areas) to reduce noise and improve OCR accuracy.

In [ ]:
width, height = page2_enhanced.size

# Adjust these fractions based on where the table actually sits
top_frac = 0.20      # <- replace with your estimate
bottom_frac = 0.75   # <- replace with your estimate

top = int(height * top_frac)
bottom = int(height * bottom_frac)

cropped = page2_enhanced.crop((0, top, width, bottom))
cropped

## Transcribe Payments table from page 2 (visual verification)
OCR on this scan remained unreliable even after rotation, contrast
enhancement, and cropping attempts. The table image is clearly legible on
visual inspection, so values are transcribed directly rather than forcing
unreliable automated extraction. This is documented as `extraction_method
= manual` with `data_quality_flag = verified`, since figures were read
directly from the source rather than inferred or OCR'd. This table uses
economic classification (Personal emoluments, Goods & services, etc.)
rather than programme classification, covering the 2025 bi-annual period
(01-Jan-2025 to 30-Jun-2025), budgeted against a full-year 2025 approved
figure.

In [ ]:
payments_rows = [
    {'sub_programme': 'Personal emoluments', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 42622728.00, 'actual_expenditure_zmw': 15581875.77, 'variance_zmw': 27040852.23, 'variance_percent': 37.0},
    {'sub_programme': 'Use of goods and services', 'budget_category': 'recurrent_expenditure',
     'approved_budget_zmw': 38064123.00, 'actual_expenditure_zmw': 16889204.68, 'variance_zmw': 21174918.32, 'variance_percent': 44.0},
    {'sub_programme': 'Financial charges', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Social benefits', 'budget_category': 'other',
     'approved_budget_zmw': 48334175.00, 'actual_expenditure_zmw': 27800800.00, 'variance_zmw': 20533375.00, 'variance_percent': 58.0},
    {'sub_programme': 'Non-financial assets', 'budget_category': 'capital_expenditure',
     'approved_budget_zmw': 52852445.00, 'actual_expenditure_zmw': 32289463.15, 'variance_zmw': 20562981.85, 'variance_percent': 61.0},
    {'sub_programme': 'Financial assets', 'budget_category': 'other',
     'approved_budget_zmw': 7473872.00, 'actual_expenditure_zmw': 326600.00, 'variance_zmw': 7147272.00, 'variance_percent': 4.0},
    {'sub_programme': 'Loan repayments', 'budget_category': 'debt_service',
     'approved_budget_zmw': 0.0, 'actual_expenditure_zmw': 0.0, 'variance_zmw': 0.0, 'variance_percent': None},
    {'sub_programme': 'Other repayments', 'budget_category': 'other',
     'approved_budget_zmw': 3478550.00, 'actual_expenditure_zmw': 3135845.14, 'variance_zmw': 342704.86, 'variance_percent': 90.0},
]

for row in payments_rows:
    row.update({
        'council_name': 'Chipata City Council',
        'budget_year': 2025,
        'budget_period': 'bi_annual',
        'vote_head_code': '',
        'programme_name': 'Total Council (Economic Classification)',
        'source_document': '2025-Bi-annual-Budget-Perfomance.pdf',
        'source_page': 2,
        'extraction_method': 'manual',
        'data_quality_flag': 'verified',
        'notes': 'Economic classification breakdown, transcribed visually from scanned Payments table (OCR unreliable)'
    })

df_payments = pd.DataFrame(payments_rows)
df_page8 = pd.concat([df_page8, df_payments], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

# verify total payments matches source
total_check = df_payments['approved_budget_zmw'].sum()
print(f"Sum of transcribed payment categories: {total_check:,.2f}")
print(f"Total payments stated in document: 192,825,893.00")
print(f"Match: {abs(total_check - 192825893.00) < 1}")
print(f"\nTotal rows in budget_allocations now: {len(df_page8)}")

## Validate required columns before export
Checking that every row has a value in all schema-required columns
(record_id, council_name, budget_year, budget_period, programme_name,
budget_category, source_document, extraction_method, extracted_at) before
final export, to catch any gaps while they're still easy to fix.

In [ ]:
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

print("Checking required columns for missing/blank values...\n")

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s):")
        print(missing[['record_id', 'programme_name', 'sub_programme']].to_string())
        print()
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete.' if all_clear else '⚠️ Some required columns need fixing before export.'}")

## Add extraction timestamp and re-validate
Populating `extracted_at` (the last required column) before final export,
then re-running the completeness check to confirm all required fields
are filled across all 60 rows.

In [ ]:
from datetime import datetime

df_page8['extracted_at'] = datetime.now().isoformat()

# re-run the check now that extracted_at exists
required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")

## Export final budget_allocations table
Saving the complete, validated table (60 rows, covering approved 2024
programme budgets and 2025 bi-annual actual payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
# ensure column order matches schema, keeping data_quality_flag as a documented extra
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

## Verify export integrity
Reading the exported CSV back in to confirm the pipe separator and encoding
work correctly and no data was lost or corrupted during export.

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." Checking
for budget/financial documents that may be available online, in addition to
the PDFs already sourced locally.

In [ ]:
!pip install requests beautifulsoup4 -q

import requests
from bs4 import BeautifulSoup

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

# List all links on the homepage
links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Web scraping — Chipata City Council website
Visiting the council's official website to fulfil the assignment's explicit
requirement to extract data "from the respective council websites." The
site returned an SSL certificate verification error (common with
government sites using outdated/misconfigured certificates) — verification
is disabled for this read-only request since no sensitive data is being
transmitted.

In [ ]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

url = "https://www.chipatacouncil.gov.zm"
response = requests.get(url, timeout=15, verify=False)
print(f"Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')
print(f"Page title: {soup.title.string if soup.title else 'No title found'}")

links = soup.find_all('a', href=True)
print(f"\nTotal links found: {len(links)}")
for link in links[:30]:
    print(link['href'], "|", link.get_text(strip=True)[:60])

## Explore Finance Department and Publications pages
Two links from the homepage appear directly relevant to budget data: the
Finance Department page and the Publications page (often where councils
post budget/financial PDF documents). Visiting both to check for content
or downloadable files relevant to budget_allocations.

In [ ]:
finance_url = "https://www.chipatacouncil.gov.zm/?page_id=5692"
pubs_url = "https://www.chipatacouncil.gov.zm/?page_id=541"

for label, page_url in [("FINANCE DEPARTMENT", finance_url), ("PUBLICATIONS", pubs_url)]:
    print(f"=== {label} ({page_url}) ===")
    resp = requests.get(page_url, timeout=15, verify=False)
    page_soup = BeautifulSoup(resp.content, 'html.parser')

    # print main visible text (rough cut, first 1000 chars)
    text = page_soup.get_text(separator=' ', strip=True)
    print(text[:1000])

    # look specifically for PDF links
    pdf_links = [a['href'] for a in page_soup.find_all('a', href=True) if '.pdf' in a['href'].lower()]
    print(f"\nPDF links found: {len(pdf_links)}")
    for pdf in pdf_links:
        print(" -", pdf)
    print("\n")

## Confirm live source and check for newer documents
Cross-referencing the council's Publications page against the PDFs already
processed confirms they are the same files hosted live on the official
website (Chipata-City-Council-budget-2024.pdf, 2025-Bi-annual-Budget-Perfomance.pdf),
satisfying the requirement to extract data from the council's website
directly. The scrape also surfaced a document not previously on hand —
budget-approved-2026.pdf — downloaded here to check whether it extends
budget coverage into a third year.

In [ ]:
new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

In [ ]:
try:
    print(f"df_page8 exists with {len(df_page8)} rows")
except NameError:
    print("df_page8 does NOT exist — needs rebuilding")

try:
    print(f"doc2026 exists with {len(doc2026)} pages")
except NameError:
    print("doc2026 does NOT exist — needs rebuilding")

In [ ]:
import pymupdf

new_budget_url = "https://www.chipatacouncil.gov.zm/wp-content/uploads/2026/05/budget-approved-2026.pdf"

resp = requests.get(new_budget_url, timeout=20, verify=False)
with open("budget_2026.pdf", "wb") as f:
    f.write(resp.content)

print(f"Downloaded: {len(resp.content)} bytes")

doc2026 = pymupdf.open("budget_2026.pdf")
print(f"Total pages: {len(doc2026)}")
print("--- PAGE 1 TEXT ---")
print(doc2026[0].get_text())

## Check scan status of 2026 budget
Page 1 returned no extractable text, and the file size (38MB for 57 pages)
is unusually large for a text-native PDF — both signs point to this being
a scanned document. Checking a few more pages to confirm before deciding
whether to pursue OCR/manual transcription or treat this as a
lower-priority addition given the size.

In [ ]:
for i in range(min(5, len(doc2026))):
    text = doc2026[i].get_text()
    print(f"=== Page {i+1} ({len(text)} chars) ===")
    print(text[:200] if text else "(no extractable text)")
    print()

## Quick check — locate summary table in 2026 budget
Before committing to full OCR across 57 pages, checking whether the 2026
budget follows the same structure as the 2024 version (programme summary
table typically appearing around page 8-9). Converting just a few candidate
pages to images and running OCR to confirm before deciding on full
extraction scope.

In [ ]:
from pdf2image import convert_from_path
import pytesseract

# Check pages 7-10 (covers the range where the table sat in the 2024 doc)
candidate_pages = convert_from_path("budget_2026.pdf", dpi=300, first_page=7, last_page=10)

for i, img in enumerate(candidate_pages):
    page_num = i + 7
    # try both orientations since we don't know if this scan is rotated
    text_normal = pytesseract.image_to_string(img)
    text_rotated = pytesseract.image_to_string(img.rotate(-90, expand=True))

    print(f"=== PAGE {page_num} (normal) ===")
    print(text_normal[:300])
    print(f"\n=== PAGE {page_num} (rotated -90) ===")
    print(text_rotated[:300])
    print("\n" + "="*50 + "\n")

## OCR the full 2026 programme budget table
Pages 8-10 confirm the same programme/sub-programme table structure as the
2024 document, with the 2024 approved figures matching exactly — confirming
this document extends the same table with 2025 and 2026 columns. Running
OCR across the full table range (pages 8-11) to capture all three years'
figures, extending the budget_allocations table with newer approved and
estimated budget data.

In [ ]:
table_pages = convert_from_path("budget_2026.pdf", dpi=400, first_page=8, last_page=12)

full_table_text = []
for i, img in enumerate(table_pages):
    text = pytesseract.image_to_string(img, config='--psm 6')
    full_table_text.append(text)
    print(f"=== PAGE {i+8} ===")
    print(text)
    print("\n")

## Extract 2025 and 2026 programme-level budgets
Table:2 on page 8 of the 2026 document is a clean summary table showing
approved budgets for 2024, 2025, and 2026 at the programme level. The 2024
column closely matches the figures already validated from the original
2024 document (minor variance likely reflects a later restatement in this
newer publication). The detailed sub-programme breakdown on pages 9-10 has
significant OCR degradation and is not extracted at this level of detail —
only the programme-level 2025 and 2026 figures are added, transcribed and
corrected from OCR against the visible table structure.

In [ ]:
# Programme-level data for 2025 and 2026, transcribed from Table:2 (page 8)
# Format: (vote_head_code, programme_name, budget_2025, budget_2026)
programme_2025_2026 = [
    ('1', 'Constituency Development', 72116301, 80065100),
    ('2', 'Local Governance', 3536881, 3773018),
    ('3', 'Integrated Development Planning', 6187954, 7173821),
    ('4', 'Economic and business development', 2110622, 2609043),
    ('5', 'Public health and Environmental protection', 4546772, 7847518),
    ('6', 'Housing and Community Amenities', 15687300, 17904197),
    ('7', 'Recreation Culture and Religion', 267068, 267069),
    ('8', 'Education and skills development', 130251, 130925),
    ('10', 'Public order and safety', 7182633, 6620906),
    ('11', 'Management and support  Services', 31284866, 31702297),
    ('12', 'Resource Mobilisation and Management', 4806840, 5535656),
    ('13', 'District Health servcies', 7790285, 8052415),
    ('15', 'Transport Services', 3200587, 6351350),
    ('16', 'Agricultural Services', 516367, 508368),
    ('17', 'Fisheries and Livestock', 486160, 503176),
    ('18', 'Social Protection and Community Development', 32975006, 13582709),
]

category_map_extended = {
    'Constituency Development': 'cdf',
    'Management and support  Services': 'recurrent_expenditure',
    'Resource Mobilisation and Management': 'recurrent_expenditure',
    'Housing and Community Amenities': 'capital_expenditure',
    'Local Governance': 'recurrent_expenditure',
    'District Health servcies': 'recurrent_expenditure',
}

new_rows = []
for code, name, amt_2025, amt_2026 in programme_2025_2026:
    for year, amt in [(2025, amt_2025), (2026, amt_2026)]:
        new_rows.append({
            'council_name': 'Chipata City Council',
            'budget_year': year,
            'budget_period': 'annual',
            'vote_head_code': code,
            'programme_name': name,
            'sub_programme': '',
            'budget_category': category_map_extended.get(name, 'other'),
            'approved_budget_zmw': amt,
            'source_document': 'budget_2026.pdf',
            'source_page': 8,
            'extraction_method': 'manual',
            'data_quality_flag': 'needs_review',
            'notes': 'Programme-level only; sub-programme detail not extracted due to OCR degradation on pages 9-10'
        })

df_new_years = pd.DataFrame(new_rows)
df_page8 = pd.concat([df_page8, df_new_years], ignore_index=True)
df_page8['record_id'] = [f"BUD-{i:03d}" for i in range(1, len(df_page8) + 1)]

print(f"Added {len(df_new_years)} new rows (2025 + 2026 programme budgets)")
print(f"Total rows now: {len(df_page8)}")

# validate against stated totals for each year
total_2025 = df_page8[(df_page8['budget_year']==2025) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
total_2026 = df_page8[(df_page8['budget_year']==2026) & (df_page8['sub_programme']=='')]['approved_budget_zmw'].sum()
print(f"\n2025 total: {total_2025:,.2f} (document shows ~192,825,892 - note likely OCR misread first digit)")
print(f"2026 total: {total_2026:,.2f} (document Head Total: 192,627,565)")

## Cross-document validation
The 2025 programme-level total extracted from the 2026 budget document
(192,825,893.00) exactly matches the "Total payments" figure independently
transcribed from the 2025 bi-annual performance report — strong
cross-document validation. The 2026 total (192,627,568.00) is within 3 ZMW
of the document's own stated Head Total (192,627,565), consistent with
rounding across 16 programmes.

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

## Export final budget_allocations table
Saving the complete, validated table (92 rows, covering approved programme
budgets for 2024, 2025, and 2026, plus actual 2025 payments by economic
classification) as the required pipe-separated CSV, following the naming
convention specified in the assignment brief.

In [ ]:
schema_columns = ['record_id', 'council_name', 'budget_year', 'budget_period', 'vote_head_code',
                   'programme_name', 'sub_programme', 'sector_cluster', 'budget_category', 'funding_source',
                   'approved_budget_zmw', 'revised_budget_zmw', 'actual_expenditure_zmw', 'variance_zmw',
                   'variance_percent', 'source_document', 'source_page', 'extraction_method',
                   'extracted_at', 'notes']

for col in schema_columns:
    if col not in df_page8.columns:
        df_page8[col] = ''

df_final = df_page8[schema_columns + ['data_quality_flag']]

df_final.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final)} rows.")
print(os.listdir())

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Rows read back: {len(df_check)}")
print(f"Columns: {list(df_check.columns)}")
df_check.head()

In [ ]:
from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")

## Remove data_quality_flag column
Dropping the data_quality_flag column to match the official schema exactly,
consistent with the other tables in the dataset. Quality-related context
for flagged rows remains preserved in the notes column where applicable.

In [ ]:
# preserve any quality flag info into notes before dropping, so nothing is lost
df_final['notes'] = df_final.apply(
    lambda row: (row['notes'] + ' | ' if row['notes'] else '') + f"[{row['data_quality_flag']}]"
    if row['data_quality_flag'] not in ('', 'verified') else row['notes'],
    axis=1
)

df_final_clean = df_final.drop(columns=['data_quality_flag'])

df_final_clean.to_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|", index=False, encoding="utf-8")
print(f"Exported {len(df_final_clean)} rows with {len(df_final_clean.columns)} columns")
print(list(df_final_clean.columns))

In [ ]:
try:
    print(f"df_page8 rows: {len(df_page8)}")
except NameError:
    print("df_page8 does NOT exist — session needs rebuilding")

try:
    print(f"doc2026 pages: {len(doc2026)}")
except NameError:
    print("doc2026 does NOT exist")

import os
print("\nFiles:", os.listdir())

In [ ]:
from datetime import datetime
df_page8['extracted_at'] = datetime.now().isoformat()

required_columns = ['record_id', 'council_name', 'budget_year', 'budget_period',
                     'programme_name', 'budget_category', 'source_document',
                     'extraction_method', 'extracted_at']

all_clear = True
for col in required_columns:
    missing = df_page8[df_page8[col].isna() | (df_page8[col].astype(str).str.strip() == '')]
    if len(missing) > 0:
        all_clear = False
        print(f"⚠️  '{col}' has {len(missing)} missing/blank row(s)")
    else:
        print(f"✅ '{col}' — no missing values")

print(f"\n{'✅ All required columns complete. Ready to export.' if all_clear else '⚠️ Fix issues before exporting.'}")
print(f"Total rows: {len(df_page8)}")

In [ ]:
df_check = pd.read_csv("db-unza26-csc4792-chipata_budget_allocations.csv", sep="|")
print(f"Final check — Rows: {len(df_check)}, Columns: {len(df_check.columns)}")

from google.colab import files
files.download("db-unza26-csc4792-chipata_budget_allocations.csv")